# **Clean**

In this section, we apply all the preprocessing steps identified during the observation and analysis phases.

The raw dataset contains images with varying dimensions, different color modes, and pixel values ranging from 0 to 255.
These inconsistencies must be addressed before training, as machine learning models require inputs that are consistent, normalized, and of fixed dimensions.

Each step below addresses a specific issue observed in the previous phases.
The cleaned dataset will be saved in a separate folder to preserve the original data and ensure reproducibility.

The following preprocessing steps will be applied:
- Removing corrupted images
- Converting all images to **RGB** format
- Resizing all images to a fixed resolution of **128×128 pixels**
- Normalizing pixel values from **0-255** to **0-1**

---

## **Corrupted Images Removal**

During the observation phase, we verified the integrity of all images in the dataset.<br>
No corrupted images were detected, meaning no files need to be excluded from the dataset.

This step is kept in the pipeline as a safeguard, in case new data is added in the future.

---

## RGB **Conversion**

Not all images in the dataset share the same color mode. Some images may be in grayscale, RGBA, or other formats, which would cause inconsistencies during training.

To ensure uniformity, all images are converted to **RGB** format, which guarantees that every image has exactly **3 color channels** (Red, Green, Blue).

Images that are already in RGB format are left unchanged.

In [ ]:
from pathlib import Path
import sys

# Auto-detect project root — works regardless of where Jupyter was launched
_search = Path.cwd()
while not (_search / "supervised").exists() and _search != _search.parent:
    _search = _search.parent

PROJECT_ROOT    = _search
SUPERVISED_ROOT = PROJECT_ROOT / "supervised"
DATASET_DIR     = str(SUPERVISED_ROOT / "dataset")
CSV_PATH        = str(SUPERVISED_ROOT / "dataset" / "dataset_info.csv")
CLEAN_DIR       = str(SUPERVISED_ROOT / "clean_dataset")
CLEAN_CSV_PATH  = str(SUPERVISED_ROOT / "clean_dataset" / "dataset_info.csv")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root : {PROJECT_ROOT}")
print(f"CSV          : {CSV_PATH}")

In [ ]:
import os
import shutil
import csv

from supervised.src.utils import plot_image_formats
from PIL import Image


def rgb_conversion(csv_path: str, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)

    with open(csv_path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)

        for row in reader:
            split = row["split"]
            class_name = row["class_name"]
            filename = row["filename"]

            input_path = str(SUPERVISED_ROOT / "dataset" / split / class_name / filename)
            output_path = f"{output_dir}/{split}/{class_name}/{filename}"

            try:
                os.makedirs(os.path.dirname(output_path), exist_ok=True)

                with Image.open(input_path) as img:
                    img = img.convert("RGB")
                    img.save(output_path)

            except Exception as e:
                print(f"Error with {input_path}: {e}")
    print(f"Conversion realized with success !")
    shutil.copy(csv_path, os.path.join(output_dir, "dataset_info.csv"))
    print(f"Copy of dataset_info.csv realized with success !")

if __name__ == '__main__':
    rgb_conversion(
        csv_path=CSV_PATH,
        output_dir=CLEAN_DIR,
    )
    plot_image_formats(
        filepath=CLEAN_CSV_PATH,
        directory="./clean_dataset",
    )

## **Image Resizing**

During the observation phase, we identified that images in the dataset have varying dimensions, with most concentrated around **500–600 pixels** and some outliers reaching significantly higher values.

This inconsistency must be addressed before training, as machine learning models require all inputs to have the same fixed dimensions.
Feeding images of different sizes would cause errors during training and could negatively impact model performance.

All images are resized to a fixed resolution of **64×64 pixels**.
This resolution was selected as a compromise between preserving sufficient visual information for the model and keeping memory usage and training time at a reasonable level.

Images that are already at the correct resolution are left unchanged to avoid unnecessary processing.

In [ ]:
import csv
import os
import matplotlib.pyplot as plt
import shutil

from PIL import Image

def plot_cleaning_stats(resized: int, skipped: int, errors: int):
    labels = ["Resized", "Already Correct", "Errors"]
    values = [resized, skipped, errors]

    plt.figure(figsize=(8,5))
    bars = plt.bar(labels, values)

    plt.title("Dataset Cleaning Summary")
    plt.ylabel("Number of Images")

    for bar, value in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width()/2, value, str(value), ha='center', va='bottom')

    plt.show()

def resize_images(csv_path: str, output_dir: str, size=(128, 128)):
    os.makedirs(output_dir, exist_ok=True)

    # Set counts for following graph
    resized_count = 0
    skipped_count = 0
    error_count = 0

    with open(csv_path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)

        for row in reader:
            split = row["split"]
            class_name = row["class_name"]
            filename = row["filename"]

            input_path = f"{output_dir}/{split}/{class_name}/{filename}"
            output_path = f"{output_dir}/{split}/{class_name}/{filename}"

            try:
                os.makedirs(os.path.dirname(output_path), exist_ok=True)

                with Image.open(input_path) as img:
                    if img.size == size:
                        img.save(output_path)
                        skipped_count += 1
                    else:
                        img = img.resize(size, Image.Resampling.LANCZOS)
                        img.save(output_path)
                        resized_count += 1

            except Exception as e:
                error_count += 1
                print(f"Error with {input_path}: {e}")
        shutil.copy(csv_path, os.path.join(output_dir, "dataset_info.csv"))
        plot_cleaning_stats(resized_count, skipped_count, error_count)

resize_images(
    csv_path=CSV_PATH,
    output_dir=CLEAN_DIR,
    size=(32, 32)
)

## **Post-Cleaning Verification**

After applying the preprocessing steps, we re-evaluate the dataset using the cleaned version.<br>
The goal of this step is to verify that all images now share consistent dimensions and that the resizing process was applied correctly.

By analyzing the distribution of image sizes on the cleaned dataset, we expect to observe a uniform resolution across all samples.

In [ ]:
from supervised.src.utils import plot_image_size_distribution

plot_image_size_distribution(CLEAN_CSV_PATH, CLEAN_DIR)

## **Pixel Normalization**

Pixel values are normalized from **0-255** to **0-1** during data loading.<br>
This step is handled directly in the data loader to avoid modifying the original images on disk.

---

# Clean — Conclusion

All preprocessing steps identified during the observation and analysis phases have been successfully applied to the dataset.

The following transformations were performed on each image:

- **Corrupted images**: No corrupted images were detected, meaning the full dataset has been preserved without any data loss.
- **RGB Conversion**: All images have been converted to RGB format, ensuring that every image has exactly 3 color channels and eliminating any inconsistencies caused by grayscale, RGBA, or other color modes.
- **Image Resizing**: All images have been resized to a fixed resolution of **128×128 pixels**, ensuring uniform input dimensions for model training.
- **Pixel Normalization**: Pixel values will be normalized from 0-255 to 0-1 during data loading, avoiding any modification of the images on disk.

The cleaned dataset has been saved in a separate `clean_dataset/` folder to preserve the integrity of the original data and ensure reproducibility.

The dataset is now fully consistent, standardized, and ready to be used for model training in the next phase.